In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests pertpy

In [ ]:
# Cell 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/perturbation_sigreg/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Upload all .py files to /content/ before running cells below:
# cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py,
# trainer.py, metrics.py, perturb_metrics.py, compare_perturbation.py,
# run_perturbation_sigreg.py

print('Drive mounted.')
print('Results will be saved to:', RESULTS_DIR)

In [ ]:
# Cell 3: Smoke test (~3 min CPU)
import subprocess, os

REPO_DIR = '/content'

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_perturbation_sigreg.py'),
     '--smoke_test', '--device', 'cpu',
     '--results_file', 'results_perturbation_sigreg_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 4: Full run — 4 conditions × ~15 min = ~60 min on A100
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_perturbation_sigreg.py'),
     '--device', 'cuda',
     '--n_epochs', '15',
     '--batch_size', '64',
     '--results_file', 'results_perturbation_sigreg.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 5: Display results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

if os.path.exists('results_perturbation_sigreg.txt'):
    print(open('results_perturbation_sigreg.txt').read())

def parse_perturb_results(path):
    if not os.path.exists(path):
        return {}
    data = {}
    for line in open(path):
        m = re.match(r'^(.{36})\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)', line)
        if m:
            name = m.group(1).strip()
            if name and not name.startswith(('=', '-', 'C')):
                data[name] = {
                    'pearson':   float(m.group(2)),
                    'pearson_d': float(m.group(3)),
                    'top20_deg': float(m.group(4)),
                    'mse':       float(m.group(5)),
                }
    return data

results = parse_perturb_results('results_perturbation_sigreg.txt')
if results:
    conditions = list(results.keys())
    metrics = ['pearson', 'pearson_d', 'top20_deg']
    metric_labels = ['Mean Pearson', 'Mean Pearson Δ', 'Top-20 DEG Pearson Δ']
    colors  = ['#4C72B0', '#4C72B0', '#DD8452', '#DD8452']
    hatches = ['', '///', '', '///']

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    x = np.arange(len(conditions))
    for ax, metric, mlabel in zip(axes, metrics, metric_labels):
        vals = [results[c][metric] for c in conditions]
        bars = ax.bar(x, vals, color=colors, alpha=0.85, width=0.6)
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        for xi, v in zip(x, vals):
            ax.text(xi, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
        ax.set_xticks(x)
        short_names = [c.replace(' (paper baseline)', '').replace(' (primary novel)', '')
                       for c in conditions]
        ax.set_xticklabels(short_names, fontsize=8, rotation=15, ha='right')
        ax.set_title(mlabel, fontsize=10)
        ax.set_ylim(0, max(vals) * 1.25 + 0.02)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)

    legend_elements = [
        Patch(facecolor='#4C72B0', label='EMA pre-training'),
        Patch(facecolor='#DD8452', label='SIGReg pre-training'),
        Patch(facecolor='white', edgecolor='black', hatch='///', label='Delta objective'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9)
    fig.suptitle('SIGReg vs EMA × Absolute vs Delta — Perturbation Prediction (Adamson 2016)',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig('perturbation_sigreg_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved perturbation_sigreg_results.png')

In [ ]:
# Cell 6: Save results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/perturbation_sigreg/'
files = [
    'results_perturbation_sigreg.txt',
    'results_perturbation_sigreg_smoke.txt',
    'perturbation_sigreg_results.png',
    'adamson2016.h5ad',
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')